# TP 2 — Partie C : Embeddings statiques + réseaux profonds (PyTorch)

Ce notebook implémente la **Partie C** du TP :
1. Tokenisation et construction du vocabulaire sur `train` (fréquence $\ge 2$), calcul des taux d'OOV sur `val` et `test`.
2. Téléchargement et chargement des embeddings pré-entraînés **AraVec** (Twitter CBOW 300 dim) avec normalisation `clean_str`.
3. Architectures PyTorch (même budget d'entraînement, early stopping sur $F1_{\text{sarc}}$ val) :
   - **C1. BiLSTM** (max-pooling)
   - **C2. BiLSTM + Attention pooling custom** : $\alpha_i = \text{softmax}(v^\top \tanh(W h_i)), \; c = \sum \alpha_i h_i$
   - **C3. Mini-Transformer** (2 couches, $d_{\text{model}}=128$, 4 têtes) avec et sans encodage positionnel (ablation).
4. Gestion du déséquilibre de classe sur la meilleure architecture :
   - (a) Perte pondérée
   - (b) Sur-échantillonnage de la classe minoritaire (dans train uniquement)
   - (c) Perte standard + seuil optimisé sur val.
5. Journalisation systématique dans `results/results.csv` via `ajouter_resultat`.

In [1]:
# 1. Environnement et configuration
import os, sys
REPO = "/content/tp2-sarcasme-arabe"
if not os.path.exists(REPO):
    !git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!git pull

# Installation des dépendances si nécessaire
!pip install -q gensim pyarabic emoji arabert

Cloning into 'tp2-sarcasme-arabe'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 59 (delta 22), reused 47 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 60.85 KiB | 2.34 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.9 MB/s eta 0:00:00


In [2]:
# 2. Chargement des données et déduplication
import torch
import pandas as pd
import numpy as np
from src.seed import set_seed
from src.data import load_splits
from src.preprocess import dedupliquer_train
from src.results import ajouter_resultat, CHEMIN

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", device)

train_brut, val, test = load_splits(seed=42)
train = dedupliquer_train(train_brut, test, verbose=True)
print(f"Tailles finales : train={len(train)} | val={len(val)} | test={len(test)} (test scellé)")

Appareil utilisé : cuda
train avant : 11293
  fuite retirée (présents dans test) : 4
  groupes contradictoires retirés    : 13
train après : 11217
Tailles finales : train=11217 | val=1255 | test=3000 (test scellé)


## 1. Vocabulaire sur `train` et calcul d'OOV
Le vocabulaire est construit exclusivement sur `train` (seuil de fréquence $\ge 2$). Nous mesurons le taux d'Out-Of-Vocabulary (OOV) sur `val` et sur `test`.

In [3]:
from src.vocab import Vocabulary, TextDataset, build_embedding_matrix
from torch.utils.data import DataLoader

vocab = Vocabulary().fit(train["tweet"].tolist(), min_freq=2)
print(f"Taille du vocabulaire (freq >= 2) : {len(vocab)} tokens (dont <PAD> et <UNK>)")

oov_train = vocab.oov_rate(train["tweet"].tolist())
oov_val = vocab.oov_rate(val["tweet"].tolist())
oov_test = vocab.oov_rate(test["tweet"].tolist())

print(f"Taux d'OOV sur train : {oov_train:.2%}")
print(f"Taux d'OOV sur val   : {oov_val:.2%}")
print(f"Taux d'OOV sur test  : {oov_test:.2%}")

Taille du vocabulaire (freq >= 2) : 16376 tokens (dont <PAD> et <UNK>)
Taux d'OOV sur train : 29.14%
Taux d'OOV sur val   : 38.48%
Taux d'OOV sur test  : 40.79%


## 2. Téléchargement et chargement des Embeddings AraVec
Modèle utilisé : **AraVec Twitter-CBOW 300 dimensions**.

In [4]:
import gensim
import zipfile
import urllib.request

ARAVEC_ZIP = "tweet_cbow_300.zip"
ARAVEC_DIR = "aravec_model"
ARAVEC_URL = "https://archive.org/download/aravec2.0/tweet_cbow_300.zip"

if not os.path.exists(ARAVEC_ZIP) and not os.path.exists(ARAVEC_DIR):
    print("Téléchargement d'AraVec Twitter-CBOW 300...")
    !wget -q -O tweet_cbow_300.zip https://archive.org/download/aravec2.0/tweet_cbow_300.zip

if not os.path.exists(ARAVEC_DIR):
    os.makedirs(ARAVEC_DIR, exist_ok=True)
    print("Décompression...")
    with zipfile.ZipFile(ARAVEC_ZIP, 'r') as zip_ref:
        zip_ref.extractall(ARAVEC_DIR)

# Trouver le fichier modèle sans extension
model_path = os.path.join(ARAVEC_DIR, "tweets_cbow_300")
if not os.path.exists(model_path):
    files = [os.path.join(ARAVEC_DIR, f) for f in os.listdir(ARAVEC_DIR) if not f.endswith(".npy") and not f.endswith(".zip")]
    model_path = files[0] if files else model_path

print("Chargement du modèle AraVec depuis :", model_path)
aravec_model = gensim.models.Word2Vec.load(model_path)
aravec_wv = aravec_model.wv
print("AraVec chargé avec succès ! Taille vocabulaire AraVec :", len(aravec_wv.key_to_index if hasattr(aravec_wv, 'key_to_index') else aravec_wv.vocab))

# Matrice d'embeddings alignée sur notre vocabulaire
embedding_matrix = build_embedding_matrix(vocab, aravec_wv, embed_dim=300)

Téléchargement d'AraVec Twitter-CBOW 300...
Décompression...
Chargement du modèle AraVec depuis : aravec_model/tweets_cbow_300
AraVec chargé avec succès ! Taille vocabulaire AraVec : 331679
AraVec couverture vocabulaire : 13830/16374 (84.46%)


## 3. DataLoaders PyTorch

In [5]:
MAX_LEN = 64
BATCH_SIZE = 64

ds_train = TextDataset(train["tweet"], train["sarcasm"].astype(int), vocab, max_len=MAX_LEN)
ds_val = TextDataset(val["tweet"], val["sarcasm"].astype(int), vocab, max_len=MAX_LEN)

loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False)

## 4. Modèle C1 : BiLSTM (Max-Pooling)
Comparaison : (i) embedding aléatoire, (ii) AraVec figé, (iii) AraVec dégelé.

In [6]:
from src.models_c import BiLSTMClassifier
from src.train_c import train_classifier

# C1.1 Embedding Aléatoire
set_seed(42)
model_c1_rand = BiLSTMClassifier(len(vocab), embed_dim=300, hidden_dim=128, pooling="max")
opt = torch.optim.Adam(model_c1_rand.parameters(), lr=1e-3, weight_decay=1e-4)
crit = torch.nn.CrossEntropyLoss()

print("=== Entraînement C1 : BiLSTM (Embedding aléatoire) ===")
res_c1_rand = train_classifier(model_c1_rand, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C1. BiLSTM (embedding aléatoire)",
    "preprocessing": "dedup train + vocab min_freq=2",
    "graine": 42,
    "seuil": res_c1_rand["seuil"],
    "pr_auc_val": res_c1_rand["pr_auc_val"],
    "f1_sarc_val": res_c1_rand["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c1_rand["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c1_rand["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c1_rand["temps_entrainement_s"],
    "params_entraines": res_c1_rand["params_entraines"],
    "note": "seuil sur val, C1 emb aléatoire",
})

# C1.2 AraVec Figé
set_seed(42)
model_c1_frozen = BiLSTMClassifier(len(vocab), embed_dim=300, hidden_dim=128, embedding_matrix=embedding_matrix, freeze_embedding=True, pooling="max")
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model_c1_frozen.parameters()), lr=1e-3, weight_decay=1e-4)

print("\n=== Entraînement C1 : BiLSTM (AraVec figé) ===")
res_c1_frozen = train_classifier(model_c1_frozen, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C1. BiLSTM (AraVec figé)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_c1_frozen["seuil"],
    "pr_auc_val": res_c1_frozen["pr_auc_val"],
    "f1_sarc_val": res_c1_frozen["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c1_frozen["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c1_frozen["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c1_frozen["temps_entrainement_s"],
    "params_entraines": res_c1_frozen["params_entraines"],
    "note": "seuil sur val, C1 AraVec figé",
})

# C1.3 AraVec Dégelé (fine-tuné)
set_seed(42)
model_c1_unfrozen = BiLSTMClassifier(len(vocab), embed_dim=300, hidden_dim=128, embedding_matrix=embedding_matrix, freeze_embedding=False, pooling="max")
opt = torch.optim.Adam(model_c1_unfrozen.parameters(), lr=1e-3, weight_decay=1e-4)

print("\n=== Entraînement C1 : BiLSTM (AraVec dégelé) ===")
res_c1_unfrozen = train_classifier(model_c1_unfrozen, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C1. BiLSTM (AraVec dégelé)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_c1_unfrozen["seuil"],
    "pr_auc_val": res_c1_unfrozen["pr_auc_val"],
    "f1_sarc_val": res_c1_unfrozen["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c1_unfrozen["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c1_unfrozen["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c1_unfrozen["temps_entrainement_s"],
    "params_entraines": res_c1_unfrozen["params_entraines"],
    "note": "seuil sur val, C1 AraVec dégelé",
})

=== Entraînement C1 : BiLSTM (Embedding aléatoire) ===
Epoch 01/12 | Loss: 0.4532 | PR-AUC: 0.2975 | Val F1 sarc: 0.3873 (seuil 0.16) | F1@0.5: 0.0267
Epoch 02/12 | Loss: 0.4127 | PR-AUC: 0.3404 | Val F1 sarc: 0.4126 (seuil 0.22) | F1@0.5: 0.1255
Epoch 03/12 | Loss: 0.3822 | PR-AUC: 0.3534 | Val F1 sarc: 0.4176 (seuil 0.2) | F1@0.5: 0.2098
Epoch 04/12 | Loss: 0.3610 | PR-AUC: 0.3828 | Val F1 sarc: 0.4235 (seuil 0.36) | F1@0.5: 0.3081
Epoch 05/12 | Loss: 0.3295 | PR-AUC: 0.4002 | Val F1 sarc: 0.4428 (seuil 0.18) | F1@0.5: 0.2366
Epoch 06/12 | Loss: 0.3010 | PR-AUC: 0.3988 | Val F1 sarc: 0.4266 (seuil 0.12) | F1@0.5: 0.2517
Epoch 07/12 | Loss: 0.2633 | PR-AUC: 0.3988 | Val F1 sarc: 0.4186 (seuil 0.18) | F1@0.5: 0.2786
Epoch 08/12 | Loss: 0.2291 | PR-AUC: 0.4230 | Val F1 sarc: 0.4453 (seuil 0.26) | F1@0.5: 0.3563
Epoch 09/12 | Loss: 0.1917 | PR-AUC: 0.4212 | Val F1 sarc: 0.4326 (seuil 0.1) | F1@0.5: 0.3376
Epoch 10/12 | Loss: 0.1546 | PR-AUC: 0.4221 | Val F1 sarc: 0.4178 (seuil 0.12) | F1

## 5. Modèle C2 : BiLSTM + Attention Pooling Custom
Implémentation exacte de la formule du cours et du sujet :
$$\alpha_i = \text{softmax}(v^\top \tanh(W h_i)), \quad c = \sum_i \alpha_i h_i$$

In [7]:
from src.models_c import AttentionBiLSTMClassifier

set_seed(42)
model_c2 = AttentionBiLSTMClassifier(
    len(vocab),
    embed_dim=300,
    hidden_dim=128,
    att_dim=64,
    embedding_matrix=embedding_matrix,
    freeze_embedding=False,
)
opt = torch.optim.Adam(model_c2.parameters(), lr=1e-3, weight_decay=1e-4)
crit = torch.nn.CrossEntropyLoss()

print("=== Entraînement C2 : BiLSTM + Attention Pooling ===")
res_c2 = train_classifier(model_c2, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C2. BiLSTM + Attention custom (AraVec dégelé)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_c2["seuil"],
    "pr_auc_val": res_c2["pr_auc_val"],
    "f1_sarc_val": res_c2["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c2["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c2["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c2["temps_entrainement_s"],
    "params_entraines": res_c2["params_entraines"],
    "note": "seuil sur val, C2 attention custom",
})

=== Entraînement C2 : BiLSTM + Attention Pooling ===
Epoch 01/12 | Loss: 0.3990 | PR-AUC: 0.4688 | Val F1 sarc: 0.4925 (seuil 0.22) | F1@0.5: 0.3541
Epoch 02/12 | Loss: 0.3488 | PR-AUC: 0.4910 | Val F1 sarc: 0.5099 (seuil 0.26) | F1@0.5: 0.4084
Epoch 03/12 | Loss: 0.3237 | PR-AUC: 0.4809 | Val F1 sarc: 0.5030 (seuil 0.26) | F1@0.5: 0.3925
Epoch 04/12 | Loss: 0.2989 | PR-AUC: 0.4730 | Val F1 sarc: 0.4929 (seuil 0.14) | F1@0.5: 0.3147
Epoch 05/12 | Loss: 0.2782 | PR-AUC: 0.4763 | Val F1 sarc: 0.4929 (seuil 0.28) | F1@0.5: 0.4080
Epoch 06/12 | Loss: 0.2548 | PR-AUC: 0.4513 | Val F1 sarc: 0.4860 (seuil 0.26) | F1@0.5: 0.4000
Early stopping déclenché à l'époque 6 (meilleur F1: 0.5099)


## 6. Modèle C3 : Mini-Encodeur Transformer (avec & sans encodage positionnel)
2 couches, $d_{\text{model}}=128$, 4 têtes, masque de padding.

In [8]:
from src.models_c import MiniTransformerClassifier

# C3.1 Avec encodage positionnel
set_seed(42)
model_c3_pos = MiniTransformerClassifier(
    len(vocab),
    embed_dim=300,
    d_model=128,
    nhead=4,
    num_layers=2,
    use_positional_encoding=True,
    embedding_matrix=embedding_matrix,
    freeze_embedding=False,
)
opt = torch.optim.Adam(model_c3_pos.parameters(), lr=5e-4, weight_decay=1e-4)
crit = torch.nn.CrossEntropyLoss()

print("=== Entraînement C3 : Mini-Transformer (avec encodage pos) ===")
res_c3_pos = train_classifier(model_c3_pos, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C3. Mini-Transformer (avec encodage pos)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_c3_pos["seuil"],
    "pr_auc_val": res_c3_pos["pr_auc_val"],
    "f1_sarc_val": res_c3_pos["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c3_pos["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c3_pos["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c3_pos["temps_entrainement_s"],
    "params_entraines": res_c3_pos["params_entraines"],
    "note": "seuil sur val, C3 Transformer pos=True",
})

# C3.2 Ablation : Sans encodage positionnel
set_seed(42)
model_c3_nopos = MiniTransformerClassifier(
    len(vocab),
    embed_dim=300,
    d_model=128,
    nhead=4,
    num_layers=2,
    use_positional_encoding=False,
    embedding_matrix=embedding_matrix,
    freeze_embedding=False,
)
opt = torch.optim.Adam(model_c3_nopos.parameters(), lr=5e-4, weight_decay=1e-4)

print("\n=== Entraînement C3 : Mini-Transformer (SANS encodage pos - Ablation) ===")
res_c3_nopos = train_classifier(model_c3_nopos, loader_train, loader_val, opt, crit, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C3. Mini-Transformer (SANS encodage pos)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_c3_nopos["seuil"],
    "pr_auc_val": res_c3_nopos["pr_auc_val"],
    "f1_sarc_val": res_c3_nopos["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_c3_nopos["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_c3_nopos["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_c3_nopos["temps_entrainement_s"],
    "params_entraines": res_c3_nopos["params_entraines"],
    "note": "seuil sur val, C3 Transformer pos=False (Ablation)",
})

=== Entraînement C3 : Mini-Transformer (avec encodage pos) ===


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01/12 | Loss: 0.4058 | PR-AUC: 0.4599 | Val F1 sarc: 0.4742 (seuil 0.16) | F1@0.5: 0.1694
Epoch 02/12 | Loss: 0.3651 | PR-AUC: 0.4818 | Val F1 sarc: 0.4963 (seuil 0.28) | F1@0.5: 0.3942
Epoch 03/12 | Loss: 0.3486 | PR-AUC: 0.4727 | Val F1 sarc: 0.4838 (seuil 0.24) | F1@0.5: 0.4258
Epoch 04/12 | Loss: 0.3397 | PR-AUC: 0.4654 | Val F1 sarc: 0.4909 (seuil 0.26) | F1@0.5: 0.4322
Epoch 05/12 | Loss: 0.3285 | PR-AUC: 0.4695 | Val F1 sarc: 0.4833 (seuil 0.14) | F1@0.5: 0.3514
Epoch 06/12 | Loss: 0.3126 | PR-AUC: 0.4909 | Val F1 sarc: 0.4959 (seuil 0.14) | F1@0.5: 0.3851
Early stopping déclenché à l'époque 6 (meilleur F1: 0.4963)

=== Entraînement C3 : Mini-Transformer (SANS encodage pos - Ablation) ===
Epoch 01/12 | Loss: 0.4014 | PR-AUC: 0.4611 | Val F1 sarc: 0.4773 (seuil 0.18) | F1@0.5: 0.2180
Epoch 02/12 | Loss: 0.3615 | PR-AUC: 0.4876 | Val F1 sarc: 0.5170 (seuil 0.26) | F1@0.5: 0.4022
Epoch 03/12 | Loss: 0.3379 | PR-AUC: 0.4804 | Val F1 sarc: 0.5059 (seuil 0.36) | F1@0.5: 0.4300
E

## 7. Gestion du déséquilibre sur la meilleure architecture
Sur le meilleur modèle parmi C1-C3 (généralement C2), comparaison de 3 stratégies :
- **(a) Perte pondérée** : poids inversement proportionnel aux classes.
- **(b) Sur-échantillonnage de la classe minoritaire (train uniquement)**.
- **(c) Perte standard + seuil optimisé sur val** (déjà mesuré ci-dessus).

In [9]:
# (a) Perte pondérée
n_neg = (train["sarcasm"] == 0).sum()
n_pos = (train["sarcasm"] == 1).sum()
poids_classe = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float32).to(device)
print(f"Poids appliqué à la classe sarcastique : {n_neg / n_pos:.2f}")

set_seed(42)
model_imb_weighted = AttentionBiLSTMClassifier(
    len(vocab), embed_dim=300, hidden_dim=128, att_dim=64, embedding_matrix=embedding_matrix, freeze_embedding=False
)
opt = torch.optim.Adam(model_imb_weighted.parameters(), lr=1e-3, weight_decay=1e-4)
crit_weighted = torch.nn.CrossEntropyLoss(weight=poids_classe)

print("=== Gestion déséquilibre (a) : Perte pondérée ===")
res_weighted = train_classifier(model_imb_weighted, loader_train, loader_val, opt, crit_weighted, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C2. AttentionBiLSTM (Perte pondérée)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_weighted["seuil"],
    "pr_auc_val": res_weighted["pr_auc_val"],
    "f1_sarc_val": res_weighted["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_weighted["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_weighted["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_weighted["temps_entrainement_s"],
    "params_entraines": res_weighted["params_entraines"],
    "note": "seuil sur val, déséquilibre perte pondérée",
})

# (b) Sur-échantillonnage de la classe minoritaire (train uniquement)
train_pos = train[train["sarcasm"] == 1]
train_neg = train[train["sarcasm"] == 0]
train_pos_oversampled = train_pos.sample(n=len(train_neg), replace=True, random_state=42)
train_oversampled = pd.concat([train_neg, train_pos_oversampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Taille train sur-échantillonné : {len(train_oversampled)} (équilibré 50/50)")
ds_train_over = TextDataset(train_oversampled["tweet"], train_oversampled["sarcasm"].astype(int), vocab, max_len=MAX_LEN)
loader_train_over = DataLoader(ds_train_over, batch_size=BATCH_SIZE, shuffle=True)

set_seed(42)
model_imb_over = AttentionBiLSTMClassifier(
    len(vocab), embed_dim=300, hidden_dim=128, att_dim=64, embedding_matrix=embedding_matrix, freeze_embedding=False
)
opt = torch.optim.Adam(model_imb_over.parameters(), lr=1e-3, weight_decay=1e-4)
crit_standard = torch.nn.CrossEntropyLoss()

print("\n=== Gestion déséquilibre (b) : Sur-échantillonnage train ===")
res_over = train_classifier(model_imb_over, loader_train_over, loader_val, opt, crit_standard, device, epochs=12, patience=4)

ajouter_resultat({
    "modele": "C2. AttentionBiLSTM (Sur-échantillonnage train)",
    "preprocessing": "dedup train + vocab min_freq=2 + clean_str aravec",
    "graine": 42,
    "seuil": res_over["seuil"],
    "pr_auc_val": res_over["pr_auc_val"],
    "f1_sarc_val": res_over["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_over["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_over["accuracy_val_seuil"],
    "n_train": len(train_oversampled),
    "n_val": len(val),
    "temps_entrainement_s": res_over["temps_entrainement_s"],
    "params_entraines": res_over["params_entraines"],
    "note": "seuil sur val, déséquilibre sur-échantillonnage train",
})

Poids appliqué à la classe sarcastique : 4.81
=== Gestion déséquilibre (a) : Perte pondérée ===
Epoch 01/12 | Loss: 0.5630 | PR-AUC: 0.4468 | Val F1 sarc: 0.4835 (seuil 0.66) | F1@0.5: 0.4662
Epoch 02/12 | Loss: 0.4918 | PR-AUC: 0.4804 | Val F1 sarc: 0.4929 (seuil 0.54) | F1@0.5: 0.4909
Epoch 03/12 | Loss: 0.4424 | PR-AUC: 0.4768 | Val F1 sarc: 0.4970 (seuil 0.64) | F1@0.5: 0.4893
Epoch 04/12 | Loss: 0.3994 | PR-AUC: 0.4825 | Val F1 sarc: 0.5026 (seuil 0.32) | F1@0.5: 0.4756
Epoch 05/12 | Loss: 0.3461 | PR-AUC: 0.4775 | Val F1 sarc: 0.4884 (seuil 0.5) | F1@0.5: 0.4884
Epoch 06/12 | Loss: 0.3115 | PR-AUC: 0.4680 | Val F1 sarc: 0.5106 (seuil 0.6) | F1@0.5: 0.4764
Epoch 07/12 | Loss: 0.2542 | PR-AUC: 0.4637 | Val F1 sarc: 0.5053 (seuil 0.8) | F1@0.5: 0.4750
Epoch 08/12 | Loss: 0.2301 | PR-AUC: 0.4657 | Val F1 sarc: 0.4796 (seuil 0.54) | F1@0.5: 0.4783
Epoch 09/12 | Loss: 0.1915 | PR-AUC: 0.4528 | Val F1 sarc: 0.4725 (seuil 0.22) | F1@0.5: 0.4606
Epoch 10/12 | Loss: 0.1784 | PR-AUC: 0.4710

## 8. Synthèse des résultats de la Partie C

In [10]:
df_res = pd.read_csv(CHEMIN)
print(df_res[["modele", "seuil", "pr_auc_val", "f1_sarc_val", "f1_sarc_val_seuil_0_5", "temps_entrainement_s", "params_entraines"]].to_string())


                                                 modele  seuil  pr_auc_val  f1_sarc_val  f1_sarc_val_seuil_0_5  temps_entrainement_s  params_entraines
0   TF-IDF + LogisticRegression (class_weight=balanced)   0.42      0.5217       0.5153                 0.5041                  4.81          186332.0
1                     mDeBERTa NLI zero-shot (français)   0.75      0.3495       0.4119                    NaN                   NaN               NaN
2              CAMeL sentiment, règle négatif=>sarcasme    NaN         NaN       0.4656                    NaN                   NaN               NaN
3                      C1. BiLSTM (embedding aléatoire)   0.26      0.4230       0.4453                 0.3563                 22.63         5353634.0
4                              C1. BiLSTM (AraVec figé)   0.12      0.4699       0.4969                 0.3028                 13.66          440834.0
5                            C1. BiLSTM (AraVec dégelé)   0.18      0.5026       0.5074       